## Time Gaps (2-14 hours) ##

(linked to Eric Bellm's SCOC 2021 Cadence Note regarding time gaps between 2-14 hours)

Transient and variability studies would prefer a logarithmic distribution of time gaps between visits, in order to study variability on all time scales. The typical cadence on the other hand, tends to place pairs of visits within a night at about 30 minutes separation and then return a few days later. Depending on the details of the cadence (in particular, if rolling cadence is implemented), this internight gap may be longer or shorter, but we do have a dearth of visits in the 2-14 hour timescales usually. 

This notebook looks at the distribution of visits acquired with longer-than-typical-pair intervals, within a night.
The metric itself can be used for any range of times.

In [ ]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

import healpy as hp
import pandas as pd

import rubin_sim.maf as maf
# import rubin_sim.utils as rsUtils

from rubin_sim.data import get_baseline

In [ ]:
# Create an output directory and connect to the current baseline simulation, available in $RUBIN_SIM_DATA_DIR

dbfile = get_baseline()

runName = os.path.split(dbfile)[-1].replace(".db", "")

print(runName)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="05_TGaps_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")
out_dir = data_dir

In [ ]:
# Look at the distribution of tgaps .. a 'big set' of Tgaps metrics for each filter and all filters is in

tgaps_dict = maf.batches.timeGaps()

list(tgaps_dict.keys())

In [ ]:
resultsDb = maf.db.ResultsDb(data_dir)

g = maf.MetricBundleGroup(tgaps_dict, dbfile, out_dir=data_dir, results_db=resultsDb)
g.run_all()

**Verified against the `rubin_sim` source** (`rubin_sim.maf.plots.special_plotters.SummaryHistogram`,
v2.6.2): this plotter's own `default_plot_dict` uses `xlabel`/`ylabel` (not `x_label`/`y_label`),
`xscale`/`yscale` (not `x_scale`/`y_scale`), and `figsize` (not `fig_size`). `legend_loc` is correct
as written -- it's read directly by `PlotHandler.plot()` itself (not by the plotter) to draw a shared
legend when several bundles are plotted together. `y_min` was already correct. The cell below has been
fixed accordingly; without this fix, the mistyped keys were silently ignored (same failure mode as the
`plot_dict` bug found earlier in `02_TDC_TimeDelayAccuracy.ipynb`), so the log-scale axes and axis
labels you asked for were never actually applied.

In [ ]:
# Plot all of the TGaps metric results (with summary histogram)
# Plot them in sets of two filters for more readable plots
for filtersets in (["u_band", "g_band"], ["r_band", "i_band"], ["z_band", "y_band"], ["all_bands", "filler"]):
    ph = maf.PlotHandler(out_dir=out_dir, fig_format="png", thumbnail=False)
    tgaps = []
    for t in tgaps_dict:
        if "Tgaps_observation" in t:
            if filtersets[0] in t or filtersets[1] in t:
                tgaps.append(tgaps_dict[t])
    ph.set_metric_bundles(tgaps)
    plot_dict = {
        "xscale": "log",
        "y_min": 0,
        "figsize": (8, 6),
        "ylabel": "Number of observation pairs",
        "xlabel": "Time gap between pairs of visits (days)",
        "yscale": "log",
        "legend_loc": (1.01, 0.5),
    }
    if filtersets[0] == "all_bands":
        plot_dict["yscale"] = None
    ph.plot(plot_func=maf.SummaryHistogram(), plot_dicts=plot_dict)

    y1, y2 = plt.ylim()
    plt.fill_between([2 / 24, 14 / 24], y1=y1, y2=y2, color="pink", alpha=0.3)
    plt.fill_between([14 / 24, 38 / 24], y1=y1, y2=y2, color="grey", alpha=0.2)
    plt.axvline(30 / 60 / 24, linestyle="--", color="k")
    plt.axvline(15 / 60 / 24, linestyle=":", color="k")
    plt.text(3 / 24.0, (y2 + y1) * 1.0 / 3.0, "2-14\n hrs", fontsize="large")
    plt.text(16 / 24.0, (y2 + y1) * 1.8 / 3, "1 d", fontsize="large")

In [ ]:
# We could look at the TGapsPercent values across the sky to see if they change significantly
tperc = []
for t in tgaps_dict:
    if "TgapsPercent" in t:
        tperc.append(tgaps_dict[t])

for t in tperc:
    ph.set_metric_bundles([t])
    ph.plot(plot_func=maf.HealpixSkyMap())

In [ ]:
tperc = []
for t in tgaps_dict:
    if "TgapsPercent_2-14hrs" in t:
        tperc.append(tgaps_dict[t])

ph.set_metric_bundles(tperc)
plot_dict = {"figsize": (8, 5), "x_min": 0, "x_max": 7, "y_max": 5000}
ph.plot(plot_func=maf.HealpixHistogram(), plot_dicts=plot_dict)

tperc = []
for t in tgaps_dict:
    if "TgapsPercent_1day" in t:
        tperc.append(tgaps_dict[t])

ph.set_metric_bundles(tperc)
plotDict = {"figsize": (8, 5), "x_min": 0, "x_max": 25}
ph.plot(plot_func=maf.HealpixHistogram(), plot_dicts=plotDict)

In [ ]:
# And we can also look at the summary statistics from each of the TGapsPercent metrics
tperc = []
for t in tgaps_dict:
    if "TgapsPercent" in t:
        tperc.append(tgaps_dict[t])

pd.DataFrame(
    [t.summary_values for t in tperc],
    index=[
        f"{t.info_label} \
                            {t.metric.name.replace('TgapsPercent_', '')}"
        for t in tperc
    ],
)

## Compare with another simulation ## 

This could be extended to work with multiple simulations, but adding even one additional simulation demonstrates the effect.

In [ ]:
# wget https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs2.0/noroll/noroll_v2.0_10yrs.db
dbfile = "noroll_v2.0_10yrs.db"
runName = os.path.split(dbfile)[-1].replace(".db", "")

print(runName)

In [ ]:
tgaps_dict2 = maf.batches.timeGaps(runName=runName)

In [ ]:
g = maf.MetricBundleGroup(tgaps_dict2, dbfile, out_dir=outDir, results_db=resultsDb)
g.run_all()

In [ ]:
# And plot both of the summary histograms of the tgaps together
ph = maf.PlotHandler(out_dir=outDir, thumbnail=False, savefig=False)

# just the 'all' bands
tgaps = []
for t in tgaps_dict:
    if "Tgaps_observation" in t and "all_bands" in t:
        tgaps.append(tgaps_dict[t])
for t in tgaps_dict2:
    if "Tgaps_observation" in t and "all_bands" in t:
        tgaps.append(tgaps_dict2[t])

ph.set_metric_bundles(tgaps)
plot_dicts = [{"color": "orange"}, {"color": "blue", "figsize": (8, 5)}]
ph.plot(plot_func=maf.SummaryHistogram(), plot_dicts=plot_dicts)

y1, y2 = plt.ylim()
plt.fill_between([2 / 24, 14 / 24], y1=y1, y2=y2, color="pink", alpha=0.3)
plt.fill_between([14 / 24, 38 / 24], y1=y1, y2=y2, color="grey", alpha=0.2)
plt.axvline(30 / 60 / 24, linestyle="--", color="k")
plt.axvline(15 / 60 / 24, linestyle=":", color="k")
plt.text(3 / 24.0, (y2 + y1) * 2 / 3.0, "2-14\n hrs", fontsize="large")
plt.text(16 / 24.0, (y2 + y1) * 1.8 / 3, "1 d", fontsize="large")

In [ ]:
# And we can also look at the summary statistics from each of the TGapsPercent metrics
tperc = []
for t in tgaps_dict2:
    if "TgapsPercent" in t:
        tperc.append(tgaps_dict2[t])

pd.DataFrame(
    [t.summary_values for t in tperc],
    index=[
        f"{t.info_label} \
                            {t.metric.name.replace('TgapsPercent_', '')}"
        for t in tperc
    ],
)